# 04 — Q1: previsão do preço do mês seguinte

**Questão:** é possível prever o preço do mês seguinte de cada produto melhor do que simplesmente repetir o preço atual?

**Método (decisões 3, 6 e 7):**
- série: preço geral (mediana do PPK entre mercados) por produto e mês;
- entradas: preços dos meses anteriores (lags) e, para Arroz, Feijão, Café e Leite, o custo do produtor (CONAB);
- modelos: baseline ingênuo, regressão linear, KNN e ensemble; o modelo extra (provavelmente Random Forest) só é decidido depois destes resultados;
- avaliação **sempre com divisão temporal** (treino nos meses antigos, teste nos recentes, ou walk-forward). **Proibido k-fold embaralhado.**
- hiperparâmetros (número de lags, k do KNN) escolhidos numa validação separada dentro do treino, nunca no teste;
- métricas em valor (MAE, MSE) e de direção sobe/não sobe (matriz de confusão, acurácia, precisão, revocação), reportadas separadamente.

Conclusão negativa (nenhum modelo bate o baseline) também é resultado e deve ser reportada com justificativa.

In [1]:
import sys
from pathlib import Path

# Permite importar o pacote src/ a partir da pasta notebooks/
sys.path.append(str(Path.cwd().parent))

import numpy as np
import pandas as pd

from src import config
import src.data_loading
import src.aggregation
import src.features
import src.evaluation
import src.q1_models

In [2]:
base_limpa = src.data_loading.carregar_base_limpa()
geral = src.aggregation.serie_geral(base_limpa)
geral.head()

,produto,mes,preco,n_mercados
0,Arroz,2022-09,4.948,29
1,Arroz,2022-10,4.969,28
2,Arroz,2022-11,4.893,27
3,Arroz,2022-12,5.044,28
4,Arroz,2023-01,5.058,28


### Separar os meses de treino antes de escolher hiperparâmetros

`escolher_hiperparametros` só pode ver os meses de treino (decisão 7: nunca
olhar o teste). `avaliar_modelo`, mais abaixo, recebe a base com **todos** os
meses — é o próprio `walk_forward` dentro dela que isola os meses de teste
(37 a 48) a partir de `config.MESES_TREINO`.

In [3]:
meses_ordenados = sorted(geral["mes"].unique())
meses_treino = meses_ordenados[:config.MESES_TREINO]
geral_treino = geral[geral["mes"].isin(meses_treino)]

len(meses_ordenados), len(meses_treino)

(48, 36)

In [4]:
linhas_resultado = []

for produto in config.PRODUTOS:
    hiperparametros = src.q1_models.escolher_hiperparametros(geral_treino, produto)
    n_lags = hiperparametros["n_lags"]
    k = hiperparametros["k"]
    pesos = src.q1_models.escolher_pesos_ensemble(geral_treino, produto, n_lags, k)

    base = src.features.montar_base_q1(geral, produto, n_lags)
    colunas_entrada = [f"lag_{i}" for i in range(1, n_lags + 1)]

    modelos = {
        "baseline": src.q1_models.BaselineIngenuo(),
        "regressao_linear": src.q1_models.criar_regressao_linear(),
        "knn": src.q1_models.criar_knn(k),
    }
    modelos["ensemble"] = src.q1_models.criar_ensemble(modelos, pesos)

    for nome_modelo, modelo in modelos.items():
        metricas = src.evaluation.avaliar_modelo(modelo, base, colunas_entrada, "alvo")
        linha = metricas.iloc[0].to_dict()
        linha.update({"produto": produto, "modelo": nome_modelo, "n_lags": n_lags, "k": k})
        linhas_resultado.append(linha)

tabela_resultados = pd.DataFrame(linhas_resultado)[
    ["produto", "modelo", "n_lags", "k", "MAE", "MSE", "acuracia", "precisao", "revocacao", "matriz_confusao"]
]
tabela_resultados.sort_values(["produto", "modelo"]).reset_index(drop=True)

,produto,modelo,n_lags,k,MAE,MSE,acuracia,precisao,revocacao,matriz_confusao
0,Arroz,baseline,6,1,0.140667,0.028151,0.333333,0.000000,0.000000,"[[2, 0], [4, 0]]"
1,Arroz,ensemble,6,1,0.155833,0.036015,0.500000,0.600000,0.750000,"[[0, 2], [1, 3]]"
2,Arroz,knn,6,1,0.277583,0.097176,0.500000,0.600000,0.750000,"[[0, 2], [1, 3]]"
3,Arroz,regressao_linear,6,1,0.150394,0.035182,0.500000,0.600000,0.750000,"[[0, 2], [1, 3]]"
4,Açúcar,baseline,6,5,0.307500,0.140337,0.666667,0.000000,0.000000,"[[4, 0], [2, 0]]"
5,Açúcar,ensemble,6,5,0.336532,0.129718,0.666667,0.500000,1.000000,"[[2, 2], [0, 2]]"
6,Açúcar,knn,6,5,0.404667,0.187513,0.500000,0.400000,1.000000,"[[1, 3], [0, 2]]"
7,Açúcar,regressao_linear,6,5,0.347487,0.154117,0.666667,0.500000,1.000000,"[[2, 2], [0, 2]]"
8,Banana Nanica,baseline,2,9,0.650000,0.688000,0.500000,0.000000,0.000000,"[[5, 0], [5, 0]]"
9,Banana Nanica,ensemble,2,9,0.652942,0.601232,0.500000,0.500000,0.200000,"[[4, 1], [4, 1]]"
